# Pre-Haiyan baseline behaviour — individual VIIRS pixels across Samar–Leyte

**Question:** Do individual pixels exhibit different baseline levels, variability and temporal behaviour before Haiyan?

Inspect **observability → mean and median absolute deviation (MAD) → baseline evolution → individual examples → candidate temporal families → spatial patterns**. All statistics and profiles are pixel-level, with no municipality or 5×5 spatial averaging. Daily observations are primary; four-day temporal medians are used only for the family model.

Use every pre-Haiyan date available in the existing Zarr, rather than assuming only 60 days. The stored dataset may still contain only a short pre-event record. A short record cannot establish annual seasonality, long-term stability, or that differences are innate. Weather-dependent sampling, mixed land use, viewing conditions and noise remain possible explanations.

This notebook is separate from Notebook 4. Its code was syntax-checked but not run against the inaccessible local datasets. Run in your Black Marble environment with xarray, Dask, rioxarray, geopandas, rasterio, Plotly, scipy and scikit-learn available. Slow data loading is isolated from plotting and clustering.


In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray
from rasterio.features import rasterize
from shapely.geometry import box
from dask.diagnostics import ProgressBar
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from IPython.display import display


In [ ]:
PROJECT_DIR = Path(
    "/Users/S4135723/Library/CloudStorage/OneDrive-RMITUniversity/"
    "02 - CH2 - Disaster Impact and Recovery/blackmarble-disaster-recovery"
)
DATA_DIR = PROJECT_DIR / "datasets"
A2_ZARR_PATH = DATA_DIR / "VNP46/processed/Haiyan_VNP46A2.zarr"
MUNICIPALITIES_PATH = DATA_DIR / "boundaries/MuniCities/MuniCities.shp"
OUTPUT_DIR = PROJECT_DIR / "output/pre_haiyan_pixel_baseline"
FIGURE_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
POI_PATH = DATA_DIR / "baseline_activity_zones.csv"  # Optional verified point inventory.
EVENT_DATE = pd.Timestamp("2013-11-08")
BASELINE_START = None  # None uses the earliest available pre-event date.
BASELINE_END = EVENT_DATE - pd.Timedelta(days=1)
DNB_BAND, MQF_BAND = "DNB_BRDF_Corrected_NTL", "Mandatory_Quality_Flag"
PROVINCES = ["Samar", "Eastern Samar", "Northern Samar", "Leyte", "Southern Leyte"]

# Exploratory screening settings, not externally validated reliability thresholds.
MIN_VALID_DAYS = 30
MIN_DAILY_COVERAGE = 0.30
MIN_MEDIAN_NTL = 1.0  # Brightness floor for ratios/clustering, not for the descriptive maps.
EDGE_WINDOW_DAYS, MIN_EDGE_OBSERVATIONS = 30, 8
STABLE_RELATIVE_MAD, STABLE_RELATIVE_SHIFT = 0.20, 0.20
AGGREGATION_DAYS = 4
MIN_OBSERVED_COMPOSITE_SHARE = 0.70
MIN_TIME_COLUMN_SHARE = 0.80
MAX_INTERNAL_GAP_BLOCKS = 2
MIN_CLUSTER_PERIODS = 12
MAX_K, FIT_SAMPLE_SIZE, SILHOUETTE_SAMPLE_SIZE = 6, 5000, 1000
RANDOM_STATE = 42
FAMILY_COLORS = px.colors.qualitative.Safe[:MAX_K]
for directory in [FIGURE_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


## 1. Establish the spatial and temporal coverage

The requested domain is the union of the five named Samar/Leyte provinces; Biliran is not included. Boundaries only delimit the study area: **no GHSL mask removes rural or dim land pixels**. Analysis is limited to the existing raster footprint. The coverage audit makes any missing regional extent explicit before “whole-region” interpretation.

Fresh observations use MQF == 0 and finite, non-negative DNB-BRDF radiance. Gap-filled NTL is not used. The recovery notebook's daily spatial P95 cap is deliberately not applied: a changing regional cap could alter the baseline variability being characterised. Bright excursions remain visible and MAD supplies a robust companion to the mean.


In [ ]:
print("Reading regional boundaries…", flush=True)
municipalities = gpd.read_file(MUNICIPALITIES_PATH)
regional_boundaries = municipalities.loc[
    municipalities["PROVINCE"].str.strip().str.title().isin(PROVINCES)
].to_crs("EPSG:4326").copy()
assert len(regional_boundaries), "No matching province boundaries."
assert set(PROVINCES).issubset(set(regional_boundaries.PROVINCE.str.strip().str.title()))
print("Opening Zarr metadata…", flush=True)
a2 = xr.open_zarr(A2_ZARR_PATH, consolidated=None, chunks="auto", mask_and_scale=True)
a2 = a2.set_coords("date").swap_dims({a2.date.dims[0]: "date"}).sortby("date")
a2 = a2.assign_coords(date=pd.to_datetime(a2.date.values).normalize())
assert a2.indexes["date"].is_unique, "Duplicate dates need resolving before analysis."
# The source dataset's saved run identifies its grid as EPSG:4326.
a2 = a2.rio.write_crs("EPSG:4326")
pre = a2.sel(date=slice(BASELINE_START, BASELINE_END))
assert pre.sizes["date"] > 0, "No pre-Haiyan observations exist in this Zarr."
calendar = pd.date_range(pd.Timestamp(pre.date.values[0]), BASELINE_END, freq="D")
print(f"Available baseline window: {calendar[0].date()} to {calendar[-1].date()} ({len(calendar)} days)")

footprint = box(*a2.rio.bounds())
coverage_geometry = regional_boundaries.copy()
coverage_geometry["geometry"] = coverage_geometry.geometry.intersection(footprint)
regional_boundaries["raster_extent_coverage_pct"] = (
    100 * coverage_geometry.to_crs("EPSG:32651").area
    / regional_boundaries.to_crs("EPSG:32651").area
)
display(regional_boundaries[["NAME_2", "PROVINCE", "raster_extent_coverage_pct"]])
if regional_boundaries.raster_extent_coverage_pct.min() < 99:
    print("PARTIAL REGION: the current raster does not cover every municipality fully.")

# Crop the grid to the domain bounding box before reading radiance.
x0, y0, x1, y1 = regional_boundaries.total_bounds
xi = np.flatnonzero((pre.x.values >= x0) & (pre.x.values <= x1))
yi = np.flatnonzero((pre.y.values >= y0) & (pre.y.values <= y1))
assert len(xi) and len(yi), "The region and raster do not overlap."
pre = pre.isel(x=slice(xi.min(), xi.max()+1), y=slice(yi.min(), yi.max()+1))
land_mask = rasterize(
    [(g, 1) for g in regional_boundaries.geometry],
    out_shape=(pre.sizes["y"], pre.sizes["x"]),
    transform=pre.rio.transform(recalc=True), fill=0, dtype="uint8",
).astype(bool)
y_index, x_index = np.where(land_mask)
pixel_ids = np.ravel_multi_index((y_index, x_index), land_mask.shape)


In [ ]:
# This is the main data-reading step. Plotting cells below reuse daily_values.
print(f"Reading {len(pixel_ids):,} pixels × {len(calendar):,} calendar days…", flush=True)
print(f"Dense radiance matrix alone: about {len(pixel_ids)*len(calendar)*4/1e6:.0f} MB; computation needs extra memory.")
pixels = pre[[DNB_BAND, MQF_BAND]].isel(
    y=xr.DataArray(y_index, dims="pixel"),
    x=xr.DataArray(x_index, dims="pixel"),
).assign_coords(pixel=pixel_ids).reindex(date=calendar)
radiance = pixels[DNB_BAND].astype("float32")
valid = ((pixels[MQF_BAND] == 0) & np.isfinite(radiance)
         & (radiance >= 0) & ~radiance.isin([6553.5, 65535.0]))
with ProgressBar():
    daily = radiance.where(valid).transpose("date", "pixel").compute()
daily_values = daily.values
observed = np.isfinite(daily_values)
print("Loaded. Subsequent sections do not reopen the raster dataset.")


## 2. Mean, MAD and changing baseline levels

The **mean** is the arithmetic mean of valid daily observations. **MAD** is the unscaled median absolute deviation from the pixel's temporal median, in radiance units. Relative MAD = MAD / median is shown only above the brightness floor.

Early-to-late change compares the medians of the first and last non-overlapping 30-day windows, each requiring at least eight observations. This is an observed baseline-level difference, not evidence of seasonality or an underlying causal trend. Cloud-conditioned sampling can alter these estimates.


In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    mean_ntl = np.nanmean(daily_values, axis=0)
    median_ntl = np.nanmedian(daily_values, axis=0)
    mad_ntl = np.nanmedian(np.abs(daily_values - median_ntl[None, :]), axis=0)
    early = daily_values[:EDGE_WINDOW_DAYS]
    late = daily_values[-EDGE_WINDOW_DAYS:]
    early_median = np.nanmedian(early, axis=0)
    late_median = np.nanmedian(late, axis=0)

n_valid = observed.sum(axis=0)
longest_gap = np.zeros(len(pixel_ids), dtype=int)
run = np.zeros(len(pixel_ids), dtype=int)
for day_valid in observed:
    run = np.where(day_valid, 0, run + 1)
    longest_gap = np.maximum(longest_gap, run)

pixel_stats = pd.DataFrame({
    "pixel_id": pixel_ids, "longitude": pre.x.values[x_index], "latitude": pre.y.values[y_index],
    "mean_ntl": mean_ntl, "median_ntl": median_ntl, "mad_ntl": mad_ntl,
    "valid_days": n_valid, "coverage_pct": 100*n_valid/len(calendar),
    "longest_gap_days": longest_gap,
    "early_valid_days": np.isfinite(early).sum(axis=0),
    "late_valid_days": np.isfinite(late).sum(axis=0),
    "early_median_ntl": early_median, "late_median_ntl": late_median,
}).set_index("pixel_id")
pixel_stats["level_supported"] = (pixel_stats.valid_days.ge(MIN_VALID_DAYS)
    & pixel_stats.coverage_pct.ge(100*MIN_DAILY_COVERAGE))
pixel_stats["bright_enough"] = pixel_stats.median_ntl.ge(MIN_MEDIAN_NTL)
pixel_stats["relative_mad"] = (pixel_stats.mad_ntl / pixel_stats.median_ntl.where(pixel_stats.bright_enough))
pixel_stats["change_supported"] = (pixel_stats.level_supported
    & pixel_stats.early_valid_days.ge(MIN_EDGE_OBSERVATIONS)
    & pixel_stats.late_valid_days.ge(MIN_EDGE_OBSERVATIONS)
    & (len(calendar) >= 2*EDGE_WINDOW_DAYS))
pixel_stats["median_change_ntl"] = (pixel_stats.late_median_ntl-pixel_stats.early_median_ntl).where(pixel_stats.change_supported)
pixel_stats["relative_change"] = (pixel_stats.median_change_ntl
    / pixel_stats.early_median_ntl.where(pixel_stats.early_median_ntl.ge(MIN_MEDIAN_NTL)))
pixel_stats["candidate_stable"] = (pixel_stats.change_supported & pixel_stats.bright_enough
    & pixel_stats.relative_mad.le(STABLE_RELATIVE_MAD)
    & pixel_stats.relative_change.abs().le(STABLE_RELATIVE_SHIFT))
pixel_stats["baseline_start"], pixel_stats["baseline_end"] = calendar[0], calendar[-1]
display(pixel_stats[["level_supported", "change_supported", "candidate_stable"]].sum().rename("pixels"))


In [ ]:
def style_figure(fig, title, height=720):
    fig.update_layout(template="plotly_white", paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="white",
        width=1280, height=height, title=dict(text=title, x=0.03),
        font=dict(family="Arial", size=14, color="#243B5A"),
        margin=dict(l=75, r=50, t=90, b=100),
        legend=dict(orientation="h", x=0, y=-0.17, xanchor="left", yanchor="top"))
    return fig

def show_save(fig, filename):
    fig.write_html(FIGURE_DIR / f"{filename}.html", include_plotlyjs="directory")
    fig.show()

boundary_x, boundary_y = [], []
for geometry in regional_boundaries.geometry:
    boundary = geometry.boundary
    for line in (list(boundary.geoms) if boundary.geom_type=="MultiLineString" else [boundary]):
        xx, yy = line.xy
        boundary_x.extend([*xx, None]); boundary_y.extend([*yy, None])

def pixel_map(values, title, colorscale="Viridis", zmin=None, zmax=None):
    grid = np.full(land_mask.shape, np.nan)
    grid[y_index, x_index] = values.reindex(pixel_ids).to_numpy(float)
    fig = go.Figure(go.Heatmap(x=pre.x.values, y=pre.y.values, z=grid,
        colorscale=colorscale, zmin=zmin, zmax=zmax, colorbar=dict(title="Value"),
        hoverongaps=False, hovertemplate="%{x:.4f}°E, %{y:.4f}°N<br>%{z:.3f}<extra></extra>"))
    fig.add_trace(go.Scatter(x=boundary_x, y=boundary_y, mode="lines",
        line=dict(color="#64748B", width=0.6), showlegend=False, hoverinfo="skip"))
    fig.update_xaxes(title="Longitude", ticksuffix="°E", range=[float(pre.x.min()), float(pre.x.max())])
    fig.update_yaxes(title="Latitude", ticksuffix="°N", range=[float(pre.y.min()), float(pre.y.max())],
        scaleanchor="x", scaleratio=1/np.cos(np.deg2rad(float(pre.y.mean()))))
    return style_figure(fig, title)

for column, title in [("mean_ntl", "Observed daily mean"), ("mad_ntl", "Temporal MAD")]:
    values = pixel_stats[column].where(pixel_stats.level_supported)
    cap = values.quantile(0.98)
    fig = pixel_map(values, f"{title} · nW cm⁻² sr⁻¹ · colour scale capped at P98", zmin=0,
                    zmax=float(cap) if pd.notna(cap) and cap>0 else None)
    show_save(fig, column)
show_save(pixel_map(pixel_stats.coverage_pct, "Observability · valid daily observations (%)", "Greens", 0, 100), "daily_coverage")
show_save(pixel_map(pixel_stats.longest_gap_days, "Longest interval without valid daily observations", "YlOrRd"), "longest_gap")
change = pixel_stats.median_change_ntl
bound = change.abs().quantile(0.98)
show_save(pixel_map(change, "Baseline evolution · late minus early median (nW cm⁻² sr⁻¹)", "RdBu_r",
                    -float(bound) if pd.notna(bound) and bound>0 else None,
                    float(bound) if pd.notna(bound) and bound>0 else None), "baseline_change")
show_save(pixel_map(pixel_stats.candidate_stable.astype(float).where(pixel_stats.change_supported & pixel_stats.bright_enough),
                    "Candidate stability · 1 = low relative MAD and small endpoint change", "Greens", 0, 1), "candidate_stability")

view = pixel_stats.loc[pixel_stats.level_supported].reset_index()
if len(view)>10000:
    view = view.sample(10000, random_state=RANDOM_STATE)
fig = px.scatter(view, x="mean_ntl", y="mad_ntl", color="coverage_pct", hover_name="pixel_id",
    hover_data=["relative_mad", "relative_change", "longest_gap_days"], color_continuous_scale="Greens",
    labels={"mean_ntl":"Mean radiance", "mad_ntl":"MAD radiance"}, opacity=0.5)
show_save(style_figure(fig, "Brightness versus variability · up to 10,000 displayed pixels"), "mean_vs_mad")


## 3. Inspect individual pixels and optional known activity zones

The examples below contrast candidate-stable, variable and changing pixels. These are data-selected examples, not externally verified land uses. Each chart retains valid daily points and a 30-calendar-day rolling mean/MAD (minimum eight observed days); unsupported windows remain blank.

To inspect airports, malls, residential areas or transport hubs, provide an optional `datasets/baseline_activity_zones.csv` with columns **name, activity_type, longitude, latitude, source, active_pre_haiyan**. Use verified coordinates and confirm that the activity existed before November 2013. Set `active_pre_haiyan` to `yes`. A point selects one containing VIIRS pixel; it does not isolate a facility or guarantee a pure land-use signal. Several zones may share the same pixel.


In [ ]:
examples = []
for label, candidates, metric, ascending in [
    ("Candidate stable", pixel_stats.loc[pixel_stats.candidate_stable], "relative_mad", True),
    ("Variable", pixel_stats.loc[pixel_stats.level_supported & pixel_stats.bright_enough], "relative_mad", False),
    ("Changing", pixel_stats.loc[pixel_stats.change_supported & pixel_stats.bright_enough].assign(
        absolute_shift=lambda frame: frame.relative_change.abs()), "absolute_shift", False),
]:
    for pid in candidates.dropna(subset=[metric]).sort_values(metric, ascending=ascending).head(2).index:
        examples.append(dict(name=f"{label} · pixel {pid}", pixel_id=pid, activity_type="Data-selected"))

if POI_PATH.exists():
    pois = pd.read_csv(POI_PATH)
    required = {"name", "activity_type", "longitude", "latitude", "source", "active_pre_haiyan"}
    assert required.issubset(pois.columns), f"Required POI fields: {sorted(required)}"
    dx = abs(float(pre.x[1]-pre.x[0])); dy = abs(float(pre.y[1]-pre.y[0]))
    for poi in pois.itertuples():
        if str(poi.active_pre_haiyan).strip().lower() != "yes":
            continue
        ix = int(np.abs(pre.x.values-poi.longitude).argmin())
        iy = int(np.abs(pre.y.values-poi.latitude).argmin())
        if abs(float(pre.x[ix])-poi.longitude)>dx/2 or abs(float(pre.y[iy])-poi.latitude)>dy/2:
            print(f"Outside available raster: {poi.name}"); continue
        pid = int(np.ravel_multi_index((iy, ix), land_mask.shape))
        if pid in pixel_stats.index:
            examples.append(dict(name=poi.name, pixel_id=pid, activity_type=poi.activity_type))
else:
    print("No verified activity-zone inventory supplied; showing data-selected pixels only.")
examples = pd.DataFrame(examples, columns=["name", "pixel_id", "activity_type"])
examples.to_csv(TABLE_DIR / "example_pixels.csv", index=False)
display(examples.merge(pixel_stats.reset_index(), on="pixel_id", how="left"))

for number, example in enumerate(examples.itertuples()):
    position = pixel_stats.index.get_loc(example.pixel_id)
    series = pd.Series(daily_values[:, position], index=calendar)
    rolling = series.rolling("30D", min_periods=MIN_EDGE_OBSERVATIONS)
    rolling_mean = rolling.mean()
    rolling_mad = rolling.apply(lambda v: np.nanmedian(np.abs(v-np.nanmedian(v))), raw=True)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
        subplot_titles=["Daily observations and rolling mean", "Rolling MAD"], vertical_spacing=0.12)
    fig.add_trace(go.Scatter(x=calendar, y=series, mode="markers", marker=dict(size=4, color="#8395A7"), name="Observed daily NTL"), row=1, col=1)
    fig.add_trace(go.Scatter(x=calendar, y=rolling_mean, mode="lines", connectgaps=False,
        line=dict(color="#174A7E", width=2), name="30-day mean"), row=1, col=1)
    fig.add_trace(go.Scatter(x=calendar, y=rolling_mad, mode="lines", connectgaps=False,
        line=dict(color="#D55E00", width=2), name="30-day MAD"), row=2, col=1)
    for row in (1,2):
        fig.add_vline(x=EVENT_DATE, line_dash="dash", line_color="#0057FF", row=row, col=1)
    fig.update_xaxes(range=[calendar[0], EVENT_DATE+pd.Timedelta(days=2)])
    fig.update_yaxes(title_text="nW cm⁻² sr⁻¹")
    show_save(style_figure(fig, f"{example.name} · {example.activity_type}"), f"pixel_example_{number:02d}")


## 4. Candidate pixel temporal families

Each row is **one pixel's temporal profile**, not a spatial average or a vector of summary metrics. Four-day medians reduce daily sampling noise. Divide by each pixel's valid-daily median and subtract one, so zero denotes its typical pre-event level. The clustering asks about relative temporal behaviour; absolute brightness remains in the family summaries.

Screen for baseline support and at least 70% observed four-day periods. Remove time columns observed in fewer than 80% of these pixels, preserving their calendar positions. Fill only complete internal gaps of at most two four-day periods in the clustering matrix; daily statistics remain untouched. Retain complete rows on the selected comparison periods, and disclose all exclusions. A large excluded fraction means these families are not representative of the whole region. Do not loosen thresholds just to obtain families.

Use a fixed random sample of at most 5,000 eligible pixel profiles to fit MiniBatchKMeans; silhouette uses at most 1,000. Predict labels for every eligible pixel. This bounds computation; it does not create spatial independence. k is selected by silhouette among non-singleton sample partitions where possible, with the elbow shown separately. Families are exploratory and conditional on missingness, normalisation and the available season.


In [ ]:
block_number = ((calendar - EVENT_DATE).days // AGGREGATION_DAYS).to_numpy()
block_ids = np.unique(block_number)
# Discard an incomplete first four-day block, if the record starts mid-block.
block_ids = block_ids[(EVENT_DATE + pd.to_timedelta(block_ids*AGGREGATION_DAYS, unit="D")) >= calendar[0]]
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    composite_values = np.stack([np.nanmedian(daily_values[block_number==b], axis=0) for b in block_ids]) if len(block_ids) else np.empty((0,len(pixel_ids)))
profiles = pd.DataFrame(composite_values.T, index=pixel_ids, columns=block_ids)
original_share = profiles.notna().mean(axis=1)
prequalified = pixel_stats.level_supported & pixel_stats.bright_enough & original_share.ge(MIN_OBSERVED_COMPOSITE_SHARE)
raw = profiles.loc[prequalified]
relative = raw.div(pixel_stats.loc[raw.index, "median_ntl"], axis=0).sub(1)
shared_blocks = relative.columns[relative.notna().mean(axis=0).ge(MIN_TIME_COLUMN_SHARE)]
# Vectorised short-gap interpolation avoids a slow loop over regional pixels.
arr = relative.to_numpy(dtype=float)
valid = np.isfinite(arr)
t = np.broadcast_to(np.arange(arr.shape[1]), arr.shape)
left = np.maximum.accumulate(np.where(valid, t, -1), axis=1)
right = np.minimum.accumulate(np.where(valid, t, arr.shape[1])[:, ::-1], axis=1)[:, ::-1]
fill = (~valid & (left >= 0) & (right < arr.shape[1])
        & ((right-left-1) <= MAX_INTERNAL_GAP_BLOCKS))
if arr.shape[1]:
    lo = np.take_along_axis(arr, np.maximum(left, 0), axis=1)
    hi = np.take_along_axis(arr, np.minimum(right, arr.shape[1]-1), axis=1)
    fraction = (t-left) / np.maximum(right-left, 1)
    arr = np.where(fill, lo + fraction*(hi-lo), arr)
prepared = pd.DataFrame(arr, index=relative.index, columns=relative.columns)
interpolation_count = pd.Series(
    fill[:, relative.columns.isin(shared_blocks)].sum(axis=1), index=relative.index
)
features = prepared.loc[:, shared_blocks].dropna()
pixel_stats["observed_composite_share"] = original_share
pixel_stats["cluster_status"] = "Insufficient baseline support or brightness"
pixel_stats.loc[pixel_stats.level_supported & pixel_stats.bright_enough, "cluster_status"] = "Insufficient four-day coverage"
pixel_stats.loc[prequalified, "cluster_status"] = "Missing values remain on selected comparison periods"
pixel_stats.loc[features.index, "cluster_status"] = "Eligible"
pixel_stats["interpolated_comparison_periods"] = interpolation_count.reindex(pixel_stats.index, fill_value=0)
pixel_stats["family_id"] = np.nan
pixel_stats["family"] = "Not grouped"
display(pixel_stats.cluster_status.value_counts().rename("pixels"))
print(f"Clustering uses {len(shared_blocks)} of {len(block_ids)} four-day periods; {len(features):,} of {len(pixel_stats):,} regional pixels qualify.")

fig = go.Figure(go.Scatter(x=EVENT_DATE+pd.to_timedelta(block_ids*4, unit="D"),
    y=100*relative.notna().mean(axis=0), mode="lines+markers", name="Observed pixel share"))
fig.add_hline(y=100*MIN_TIME_COLUMN_SHARE, line_dash="dot")
fig.add_vline(x=EVENT_DATE, line_dash="dash", line_color="#0057FF")
fig.update_yaxes(title="Observed prequalified pixels (%)", range=[0,100])
show_save(style_figure(fig, "Which pre-event periods are shared across pixel profiles?"), "clustering_time_support")


In [ ]:
chosen_k, elbow_k = None, None
models, rows = {}, []
# Require comparison periods across the record, not solely near its end.
thirds = np.array_split(block_ids, 3)
broad_support = all(len(np.intersect1d(shared_blocks, stage)) >= 2 for stage in thirds)
if len(features) >= 4 and len(shared_blocks) >= MIN_CLUSTER_PERIODS and broad_support:
    training = features.sample(min(FIT_SAMPLE_SIZE, len(features)), random_state=RANDOM_STATE)
    X = training.to_numpy(dtype="float32")
    unique_count = len(np.unique(X, axis=0))
    rng = np.random.default_rng(RANDOM_STATE)
    score_indices = rng.choice(len(X), min(SILHOUETTE_SAMPLE_SIZE, len(X)), replace=False)
    for k in range(1, min(MAX_K, len(X)-1, unique_count)+1):
        model = MiniBatchKMeans(n_clusters=k, n_init=10, batch_size=1024, random_state=RANDOM_STATE).fit(X)
        labels = model.predict(X)
        scored_labels = labels[score_indices]
        n_labels = len(np.unique(scored_labels))
        score = silhouette_score(X[score_indices], scored_labels) if 1<n_labels<len(score_indices) else np.nan
        counts = np.bincount(labels, minlength=k)
        rows.append(dict(k=k, inertia=model.inertia_, silhouette=score, minimum_family_size=int(counts.min())))
        models[k] = model
    diagnostics = pd.DataFrame(rows, columns=["k","inertia","silhouette","minimum_family_size"])
    scored = diagnostics.dropna(subset=["silhouette"])
    preferred = scored.loc[scored.minimum_family_size.ge(2)]
    if not scored.empty:
        chosen_k = int((preferred if not preferred.empty else scored).sort_values(["silhouette","k"], ascending=[False,True]).iloc[0].k)
    if len(diagnostics)>=3 and diagnostics.inertia.iloc[0]>diagnostics.inertia.iloc[-1]:
        xx = (diagnostics.k-1)/(diagnostics.k.iloc[-1]-1)
        yy = (diagnostics.inertia-diagnostics.inertia.iloc[-1])/(diagnostics.inertia.iloc[0]-diagnostics.inertia.iloc[-1])
        departure = ((1-xx)-yy).iloc[1:-1]
        if departure.max()>1e-9:
            elbow_k = int(diagnostics.loc[departure.idxmax(),"k"])
else:
    diagnostics = pd.DataFrame(columns=["k","inertia","silhouette","minimum_family_size"])

if chosen_k is not None:
    model = models[chosen_k]
    # Stable numeric ordering by centre variability; numbers have no causal meaning.
    order = np.argsort(np.std(model.cluster_centers_, axis=1), kind="stable")
    label_map = {int(old):new for new,old in enumerate(order)}
    predicted = model.predict(features.to_numpy(dtype="float32"))
    family_ids = np.array([label_map[int(label)] for label in predicted])
    pixel_stats.loc[features.index,"family_id"] = family_ids
    pixel_stats.loc[features.index,"family"] = [f"F{i+1}" for i in family_ids]
    pixel_stats.loc[features.index,"cluster_status"] = "Grouped"
    print(f"Selected k={chosen_k}; elbow={elbow_k}; inspect disagreement and family sizes.")
else:
    pixel_stats.loc[features.index,"cluster_status"] = "Not fitted: insufficient comparable profiles, time support, or valid silhouette"
    print("No defensible multi-family fit under the current settings. Descriptive maps and daily profiles remain usable.")
fig = make_subplots(rows=1, cols=2, subplot_titles=["Inertia / elbow", "Mean silhouette"])
for col,metric in [(1,"inertia"),(2,"silhouette")]:
    fig.add_trace(go.Scatter(x=diagnostics.k, y=diagnostics[metric], mode="lines+markers", showlegend=False), row=1,col=col)
    if chosen_k is not None:
        fig.add_vline(x=chosen_k, line_dash="dash", line_color="#D55E00", row=1,col=col)
if elbow_k is not None:
    fig.add_vline(x=elbow_k, line_dash="dot", line_color="#009E73", row=1,col=1)
fig.update_xaxes(title="Candidate k", dtick=1)
show_save(style_figure(fig, "Pixel temporal families · orange: selected; green: elbow"), "family_k_diagnostics")


In [ ]:
family_summary = pd.DataFrame()
if chosen_k is not None:
    fig = go.Figure()
    for family in range(chosen_k):
        members = pixel_stats.index[pixel_stats.family_id.eq(family)]
        values = features.loc[members]
        # Reinsert omitted periods so curves cannot imply continuity across them.
        for pid in values.sample(min(12,len(values)), random_state=RANDOM_STATE).index:
            curve = relative.loc[pid].reindex(block_ids)
            fig.add_trace(go.Scatter(x=EVENT_DATE+pd.to_timedelta(block_ids*4+1.5, unit="D"),
                y=100*curve, mode="lines", connectgaps=False, opacity=0.18,
                line=dict(color=FAMILY_COLORS[family], width=1), showlegend=False,
                name=f"F{family+1} · pixel {pid}"))
        centre = values.median().reindex(block_ids)
        fig.add_trace(go.Scatter(x=EVENT_DATE+pd.to_timedelta(block_ids*4+1.5, unit="D"),
            y=100*centre, mode="lines+markers", connectgaps=False,
            line=dict(color=FAMILY_COLORS[family], width=3), name=f"F{family+1}: {len(members):,} pixels"))
    fig.add_hline(y=0, line_dash="dot", line_color="#6C7882")
    fig.add_vline(x=EVENT_DATE, line_dash="dash", line_color="#0057FF")
    fig.update_yaxes(title="Deviation from pixel baseline median (%)")
    show_save(style_figure(fig, "Original pixel examples and prepared-family medians"), "family_profiles")
    colours = ["#B8BEC5"] + FAMILY_COLORS[:chosen_k]
    scale = []
    for i,colour in enumerate(colours):
        scale.extend([[i/len(colours),colour],[(i+1)/len(colours),colour]])
    map_values = pixel_stats.family_id.add(1).fillna(0)
    fig = pixel_map(map_values, "Pixel baseline temporal families · grey = not grouped", scale, -0.5, chosen_k+0.5)
    fig.data[0].colorbar.update(tickvals=list(range(chosen_k+1)), ticktext=["Not grouped"]+[f"F{i+1}" for i in range(chosen_k)])
    show_save(fig, "pixel_family_map")
    family_summary = pixel_stats.loc[pixel_stats.family_id.notna()].groupby("family").agg(
        pixels=("family_id","size"), median_mean_ntl=("mean_ntl","median"), median_mad_ntl=("mad_ntl","median"),
        median_relative_mad=("relative_mad","median"), median_coverage_pct=("coverage_pct","median"),
        median_change=("relative_change","median"), candidate_stable_share=("candidate_stable","mean"))
    display(family_summary)


## 5. Interpretation and next data collection

A bright pixel with low relative MAD and little supported endpoint change is a **candidate stable reference within this observed window**. It is not necessarily an invariant reference across seasons. Mean and MAD maps include sufficiently observed dim pixels; ratio-based screening and families use a brightness floor to avoid dividing by near-zero baselines.

Compare family radiance, variability, baseline change and observability together. If labels align mainly with missingness, they may reflect sampling rather than different activity. Airport, mall, residential and transport labels require a verified pre-2013 inventory and remain mixed-pixel interpretations. Family membership does not establish an inherent property of a pixel.

For stronger baseline characterisation, extend the same VNP46A2 store back to the earliest available product observations before Haiyan, maintaining the same grid and QA rules. Aim for at least a full annual cycle where available and inspect comparable seasons, change points and calendar effects before interpreting long-term stability. VNP46A2 does not provide an unlimited historical pre-Haiyan record; do not treat post-Haiyan recovery years as undisturbed baseline. With a longer record, revisit the exploratory thresholds rather than assuming they transfer unchanged.

The immediate output is an observed baseline atlas and candidate pixel families, with explicit observation support. Physical/life-quality comparison remains a later stage.


In [ ]:
pixel_stats.to_csv(TABLE_DIR / "pixel_baseline_statistics_and_families.csv")
diagnostics.to_csv(TABLE_DIR / "pixel_family_k_diagnostics.csv", index=False)
family_summary.to_csv(TABLE_DIR / "pixel_family_summary.csv")
profiles.to_csv(TABLE_DIR / "pixel_four_day_profiles.csv")
features.to_csv(TABLE_DIR / "pixel_clustering_features.csv")
regional_boundaries[["NAME_2","PROVINCE","raster_extent_coverage_pct"]].to_csv(TABLE_DIR / "regional_extent_audit.csv", index=False)
settings = dict(signal=DNB_BAND, mqf=0, spatial_support="individual land pixels; no GHSL filter",
    provinces=PROVINCES, baseline_start=str(calendar[0].date()), baseline_end=str(calendar[-1].date()),
    daily_spatial_cap=None, mad_definition="median absolute deviation from temporal median; unscaled",
    min_valid_days=MIN_VALID_DAYS, min_daily_coverage=MIN_DAILY_COVERAGE, min_median_ntl=MIN_MEDIAN_NTL,
    edge_window_days=EDGE_WINDOW_DAYS, min_edge_observations=MIN_EDGE_OBSERVATIONS,
    stable_relative_mad=STABLE_RELATIVE_MAD, stable_relative_shift=STABLE_RELATIVE_SHIFT,
    aggregation_days=AGGREGATION_DAYS, min_observed_composite_share=MIN_OBSERVED_COMPOSITE_SHARE,
    min_time_column_share=MIN_TIME_COLUMN_SHARE, max_internal_gap_blocks=MAX_INTERNAL_GAP_BLOCKS,
    min_cluster_periods=MIN_CLUSTER_PERIODS, clustering_normalisation="pixel daily median ratio minus one",
    random_state=RANDOM_STATE, fit_sample_size=FIT_SAMPLE_SIZE, silhouette_sample_size=SILHOUETTE_SAMPLE_SIZE,
    selected_blocks=[int(b) for b in shared_blocks], chosen_k=chosen_k, elbow_k=elbow_k)
(TABLE_DIR / "baseline_settings.json").write_text(json.dumps(settings, indent=2))
print("Outputs:", OUTPUT_DIR)
